# CS Lifeline — research pipeline (early-warning model)

This notebook runs the **same** pipeline as the command line (`python -m cslifeline_ml train`), step by step, so each decision can be inspected.

* **Data**: either the bundled **synthetic** cohorts (default) or a **research export** downloaded from the teacher portal (pseudonymous IDs only).
* **Label**: assessment accuracy in the 28 days *after* a cut-off is below 50 %. Features use only records *before* the cut-off.
* **Validation**: learner-grouped 5-fold cross-validation on the training cohort, then one evaluation on a *later* cohort of *different* learners.

> Results on synthetic data test the software only. They are **not** evidence about real learners and must not be reported as model accuracy in the thesis.

## 1. Set-up
In Google Colab, upload `cslifeline_ml.zip` (the `ml/` folder from the CS Lifeline ZIP) when asked, or set `REPO_URL` to a Git repository containing it. Running locally from the `ml/` folder needs no upload.

In [ ]:
import os, sys, pathlib
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = ''   # optional: e.g. https://github.com/<you>/cslifeline.git
if IN_COLAB:
    if REPO_URL:
        !git clone -q $REPO_URL /content/cslifeline
        os.chdir('/content/cslifeline/ml')
    else:
        from google.colab import files
        up = files.upload()            # choose cslifeline_ml.zip
        !unzip -q -o cslifeline_ml.zip -d /content/ml_pkg
        os.chdir('/content/ml_pkg/ml' if pathlib.Path('/content/ml_pkg/ml').exists() else '/content/ml_pkg')
    !pip -q install scikit-learn pandas matplotlib joblib xgboost
sys.path.insert(0, os.getcwd())
print('Working in', os.getcwd())

## 2. Choose the data
`DATA = 'synthetic'` uses the generated cohorts H1 (training), H2 (temporal hold-out) and C (current term). For a real study, upload the research-export CSV files to a folder and set `DATA` to that folder, `REAL = True` only if the data is real and consented, and choose the cohorts (add a `cohort` column to `learners.csv`, `responses.csv` and `sessions.csv` if you have more than one term).

In [ ]:
DATA = 'synthetic'      # or a folder path with learners.csv, responses.csv, sessions.csv
REAL = False
TRAIN_COHORT, TEST_COHORT = 'H1', 'H2'
from cslifeline_ml.dataset import load_synthetic, load_export, build_table, FEATURE_SETS
data = load_synthetic() if DATA == 'synthetic' else load_export(DATA)
{k: len(v) for k, v in data.items()}

## 3. Build leakage-safe tables
One row per learner per cut-off (after week 5 and after week 8 of term).

In [ ]:
train = build_table(data, [TRAIN_COHORT]); test = build_table(data, [TEST_COHORT])
for name, t in (('train', train), ('test', test)):
    lab = t[t['label'].notna()]
    print(f"{name}: {len(t)} rows, {lab.shape[0]} labelled, prevalence {lab['label'].mean():.1%}")
train.head()

## 4. Exploratory analysis

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'serif'})
lab = train[train['label'].notna()]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, f in zip(ax, ['acc_all', 'diag_error_rate', 'active_days_28']):
    for y, ls in ((0, '-'), (1, '--')):
        a.hist(lab[lab['label'] == y][f].dropna(), bins=20, histtype='step', linestyle=ls, color='black', density=True, label=f'label={y}')
    a.set_title(f); a.legend()
plt.tight_layout(); plt.show()
lab[FEATURE_SETS['cslifeline'] + ['label']].corr()['label'].sort_values()

## 5. Train, compare and evaluate

In [ ]:
from pathlib import Path
from cslifeline_ml.dataset import SYN
from cslifeline_ml.pipeline import train_and_evaluate, save_to_registry, labelled
files = sorted(SYN.glob('*.csv*')) if DATA == 'synthetic' else sorted(Path(DATA).glob('*.csv'))
res = train_and_evaluate(train, test, source='synthetic' if DATA == 'synthetic' else ('real' if REAL else 'export-unverified'), data_files=files, train_label=TRAIN_COHORT, test_label=TEST_COHORT)
e = res['entry']
import pandas as pd
pd.DataFrame({k: {'CV PR-AUC': v['pr_auc'], 'CV ROC-AUC': v['roc_auc']} for k, v in e['metrics']['cv'].items()}).T.join(
  pd.DataFrame({k: {'hold-out PR-AUC': v['pr_auc'], 'hold-out recall': v.get('recall'), 'hold-out precision': v.get('precision'), 'Brier': v['brier']} for k, v in e['metrics']['test_all_models'].items()}).T).round(3)

In [ ]:
print('Chosen:', e['model_name'], '| thresholds', e['thresholds'])
t = e['metrics']['test']
print(f"Hold-out: PR-AUC {t['pr_auc']:.3f} (baseline {t['prevalence']:.3f}), ROC-AUC {t['roc_auc']:.3f}, recall {t['recall']:.2f}, precision {t['precision']:.2f}")
print('Confusion at flag threshold:', t['flag']['confusion'])
print('Ablation (behaviour+engagement):', e['metrics']['ablation']['pr_auc_gain_of_behaviour_and_engagement'])
print('Cross-platform (combined − CS Lifeline):', e['metrics']['cross_platform']['combined_minus_cslifeline'])
if e.get('warning'): print('\nWARNING:', e['warning'])

## 6. Register, report and predict
The model is registered as a **candidate**. Promotion to production is a deliberate, recorded step (Administration → Models in the app, or `python -m cslifeline_ml promote`).

In [ ]:
from cslifeline_ml.pipeline import REPORTS_DIR, MODELS_DIR
from cslifeline_ml.report import write_report
d = save_to_registry(res)
labelled(train)[e['features']].mean().to_json(d / 'train_means.json')
print('Registered in', d)
print('Report:', write_report(res, REPORTS_DIR))

Upload `ml/models/<version>/registry_entry.json` in **Administration → Models → Register model**, promote it there with a reason, then run the next cell and import the predictions file with **Import predictions**.

In [ ]:
from cslifeline_ml.__main__ import main
if DATA == 'synthetic':
    main(['predict', '--model', e['model_version'], '--cohort', 'C', '--as-of', '2026-11-05T12:00:00Z'])